In [ ]:
import os
from pathlib import Path

import geopandas as gpd
import pandas as pd
import rasterio

root = ""
data_folder = root + "../burnp3plus/hex16"
spatial_dir = data_folder + "/spatial/"

spatial_data = {"rasters": {}, "vectors": {}}
tabular_data = {}

### Spatial

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.plot import plotting_extent

spatial_dir = data_folder + "/spatial/"

path = spatial_dir + "hex16_dem.tif"
with rasterio.open(path) as src:
    data = src.read(1, masked=True)  # mask out the nodata (-9999 values)
    profile = src.profile  # Metadata (cell size, CRS, bounds, etc.)
    extent = plotting_extent(src)  # to position it into real-world coordinates


print("Min:", data.min())
print("Max:", data.max())
print("Bounds:", src.bounds)
print("Transform:", src.transform)
print("Resolution:", src.res)
print("CRS:", src.crs)

plt.figure(figsize=(10, 10))
img = plt.imshow(
    data,
    cmap="viridis",
    extent=extent,
    aspect="equal",
    origin="upper",  # Flip correctly for north-up rasters
)
plt.colorbar(img, label="DEM")
plt.title("DEM")
plt.xlabel("Easting (m)")
plt.ylabel("Northing (m)")
plt.show()

####

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.io import MemoryFile
from rasterio.plot import plotting_extent
from rasterio.warp import Resampling, calculate_default_transform, reproject


def reproject_and_plot_to_esri_102002(
    input_path: str,
    band: int = 1,
    resampling: Resampling = Resampling.bilinear,
    cmap: str = "viridis",
    figsize: tuple[int, int] = (10, 10),
    title: str | None = None,
):
    """
    Reproject a raster to ESRI:102002 in memory and plot it without saving.
    """
    dst_crs = "ESRI:102002"

    with rasterio.open(input_path) as src:
        transform, width, height = calculate_default_transform(src.crs, dst_crs, src.width, src.height, *src.bounds)

        profile = src.profile.copy()
        profile.update(crs=dst_crs, transform=transform, width=width, height=height)

        with MemoryFile() as memfile:
            with memfile.open(**profile) as dst:
                for i in range(1, src.count + 1):
                    reproject(
                        source=rasterio.band(src, i),
                        destination=rasterio.band(dst, i),
                        src_transform=src.transform,
                        src_crs=src.crs,
                        dst_transform=transform,
                        dst_crs=dst_crs,
                        src_nodata=src.nodata,
                        dst_nodata=src.nodata,
                        resampling=resampling,
                    )

                data = dst.read(band, masked=True)
                extent = plotting_extent(data, dst.transform)

                plt.figure(figsize=figsize)
                img = plt.imshow(
                    data,
                    cmap=cmap,
                    extent=extent,
                    origin="upper",
                    aspect="equal",
                )
                plt.colorbar(img, label="Value")
                plt.title(title or f"Reprojected to {dst_crs}")
                plt.xlabel("Easting (m)")
                plt.ylabel("Northing (m)")
                plt.show()

                print("Min:", data.min())
                print("Max:", data.max())
                print("Bounds:", dst.bounds)
                print("Transform:", dst.transform)
                print("Resolution:", dst.res)
                print("CRS:", dst.crs)


from rasterio.warp import Resampling

dem_path = spatial_dir + "hex05_dem.tif"

reproject_and_plot_to_esri_102002(
    input_path=dem_path, band=1, resampling=Resampling.bilinear, cmap="terrain", title="DEM reprojected to ESRI:102002"
)

In [ ]:
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show

with rasterio.open(path) as src:
    fig, ax = plt.subplots(figsize=(10, 10))
    show(src.read(1, masked=True), transform=src.transform, ax=ax, cmap="viridis")
    ax.set_title("DEM")
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    plt.show()

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show

dem_path = spatial_dir + "hex05_dem.tif"
actual_path = spatial_dir + "mask_grids/hex05_actual.shp"
buffer_path = spatial_dir + "mask_grids/hex05_buffer.shp"

with rasterio.open(dem_path) as src:
    dem = src.read(1, masked=True)
    dem_crs = src.crs
    transform = src.transform

actual = gpd.read_file(actual_path).to_crs(dem_crs)
buffer = gpd.read_file(buffer_path).to_crs(dem_crs)

fig, ax = plt.subplots(figsize=(10, 10))
show(dem, transform=transform, ax=ax, cmap="viridis")
actual.boundary.plot(ax=ax, color="red", linewidth=2, label="actual")
buffer.boundary.plot(ax=ax, color="yellow", linewidth=2, label="buffer")

with rasterio.open(dem_path) as src:
    print("DEM bounds:", src.bounds)

actual = gpd.read_file(actual_path).to_crs(dem_crs)
buffer = gpd.read_file(buffer_path).to_crs(dem_crs)

print("Actual bounds:", actual.total_bounds)  # [minx, miny, maxx, maxy]
print("Buffer bounds:", buffer.total_bounds)

ax.legend()
ax.set_title("DEM with actual and buffer masks")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.show()

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.mask import mask
from rasterio.plot import plotting_extent, show


def plot_raster_with_masks(
    raster_path: str,
    actual_mask_path: str | None = None,
    buffer_mask_path: str | None = None,
    band: int = 1,
    cmap: str = "viridis",
    figsize: tuple[int, int] = (10, 10),
    title: str | None = None,
):
    """
    Plot a raster and optionally overlay actual/buffer mask boundaries.
    """
    with rasterio.open(raster_path) as src:
        data = src.read(band, masked=True)
        raster_crs = src.crs
        transform = src.transform
        bounds = src.bounds
        nodata = src.nodata

    fig, ax = plt.subplots(figsize=figsize)
    show(data, transform=transform, ax=ax, cmap=cmap)

    if actual_mask_path is not None:
        actual_gdf = gpd.read_file(actual_mask_path).to_crs(raster_crs)
        actual_gdf.boundary.plot(ax=ax, color="red", linewidth=2, label="actual")

    if buffer_mask_path is not None:
        buffer_gdf = gpd.read_file(buffer_mask_path).to_crs(raster_crs)
        buffer_gdf.boundary.plot(ax=ax, color="yellow", linewidth=2, label="buffer")

    ax.set_title(title or f"Raster with masks\n{raster_path}")
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")

    handles, labels = ax.get_legend_handles_labels()
    if labels:
        ax.legend()

    plt.show()

    print("Raster Min:", np.nanmin(data))
    print("Raster Max:", np.nanmax(data))
    print("Raster bounds:", bounds)
    print("Raster CRS:", raster_crs)
    print("Raster transform:", transform)
    print("Raster nodata:", nodata)


def clip_raster_to_mask(
    raster_path: str,
    mask_path: str,
    output_path: str | None = None,
    band: int = 1,
    crop: bool = True,
    filled: bool = False,
    cmap: str = "viridis",
    figsize: tuple[int, int] = (10, 10),
    title: str | None = None,
):
    """
    Clip a raster to a polygon mask, optionally save it, and plot it.
    """
    mask_gdf = gpd.read_file(mask_path)

    with rasterio.open(raster_path) as src:
        mask_gdf = mask_gdf.to_crs(src.crs)

        out_image, out_transform = mask(src, mask_gdf.geometry, crop=crop, filled=filled)

        out_meta = src.meta.copy()
        out_meta.update(
            {
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
            }
        )

        nodata = src.nodata

    data = out_image[band - 1]

    if filled and nodata is not None:
        data = np.ma.masked_equal(data, nodata)

    extent = plotting_extent(data, out_transform)

    plt.figure(figsize=figsize)
    img = plt.imshow(
        data,
        cmap=cmap,
        extent=extent,
        origin="upper",
        aspect="equal",
    )
    plt.colorbar(img, label="Value")
    plt.title(title or f"Clipped raster\n{raster_path}")
    plt.xlabel("Easting (m)")
    plt.ylabel("Northing (m)")
    plt.show()

    if output_path is not None:
        with rasterio.open(output_path, "w", **out_meta) as dst:
            dst.write(out_image)
        print(f"Saved clipped raster to: {output_path}")

    return out_image, out_transform, out_meta


clipped_dem, clipped_transform, clipped_meta = clip_raster_to_mask(
    raster_path=dem_path, mask_path=buffer_path, filled=True, title="DEM clipped to actual mask"
)

In [ ]:
firezones_path = spatial_dir + "hex05_firezones.tif"


clipped_dem, clipped_transform, clipped_meta = clip_raster_to_mask(
    raster_path=firezones_path, mask_path=actual_path, filled=True, title="Fire Zones clipped to actual mask"
)

In [ ]:
firezones_path = spatial_dir + "hex05_fbp.tif"


clipped_dem, clipped_transform, clipped_meta = clip_raster_to_mask(
    raster_path=firezones_path, mask_path=actual_path, filled=True, title="FBP clipped to actual mask"
)

In [ ]:
firezones_path = spatial_dir + "ignition_grids/hex05_ignGrid_N_s1.tif"


clipped_dem, clipped_transform, clipped_meta = clip_raster_to_mask(
    raster_path=firezones_path, mask_path=actual_path, filled=True, title="Ignition Grid clipped to actual mask"
)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.mask import mask
from rasterio.plot import plotting_extent


def sum_rasters(raster_paths: list[str], output_path: str | None = None):
    arrays = []
    meta = None
    ref_shape = None
    ref_transform = None
    ref_crs = None
    nodata = None

    for path in raster_paths:
        with rasterio.open(path) as src:
            data = src.read(1, masked=True).astype(np.float32)

            if meta is None:
                meta = src.meta.copy()
                ref_shape = data.shape
                ref_transform = src.transform
                ref_crs = src.crs
                nodata = src.nodata
            else:
                if data.shape != ref_shape:
                    raise ValueError(f"Shape mismatch in {path}")
                if src.transform != ref_transform:
                    raise ValueError(f"Transform mismatch in {path}")
                if src.crs != ref_crs:
                    raise ValueError(f"CRS mismatch in {path}")

            arrays.append(data.filled(0))

    summed = np.sum(arrays, axis=0)

    if output_path is not None:
        meta.update(dtype="float32", count=1, nodata=0)
        with rasterio.open(output_path, "w", **meta) as dst:
            dst.write(summed.astype(np.float32), 1)
        print(f"Saved summed raster to: {output_path}")

    return summed, meta


raster_paths = [
    spatial_dir + "ignition_grids/hex05_ignGrid_N_s1.tif",
    spatial_dir + "ignition_grids/hex05_ignGrid_N_s2.tif",
    spatial_dir + "ignition_grids/hex05_ignGrid_H_s1.tif",
    spatial_dir + "ignition_grids/hex05_ignGrid_H_s2.tif",
]

# Step 1: sum and save
summed, meta = sum_rasters(
    raster_paths,
)

clipped_dem, clipped_transform, clipped_meta = clip_raster_to_mask(
    raster_path=firezones_path, mask_path=actual_path, filled=True, title="Summed Ignition Grid clipped to actual mask"
)

In [ ]:
# Tabular Data:
from pathlib import Path

import pandas as pd


def read_tabular_folder(
    tabular_dir: str | Path,
    pattern: str = "*.csv",
    encoding: str | None = None,
    verbose: bool = True,
) -> dict[str, pd.DataFrame]:
    """
    Read all CSV files in a folder into a dictionary of DataFrames.

    Returns:
        dict where:
        - key   = file stem without extension
        - value = pandas DataFrame
    """
    tabular_dir = Path(tabular_dir)

    if not tabular_dir.exists():
        raise FileNotFoundError(f"Folder does not exist: {tabular_dir}")

    if not tabular_dir.is_dir():
        raise NotADirectoryError(f"Path is not a directory: {tabular_dir}")

    csv_files = sorted(tabular_dir.glob(pattern))

    if not csv_files:
        raise FileNotFoundError(f"No files matching '{pattern}' found in {tabular_dir}")

    tables: dict[str, pd.DataFrame] = {}

    for csv_path in csv_files:
        try:
            df = pd.read_csv(csv_path, encoding=encoding) if encoding else pd.read_csv(csv_path)
            tables[csv_path.stem] = df

            if verbose:
                print(f"Loaded: {csv_path.name:40s} shape={df.shape}")

        except Exception as e:
            print(f"Failed to read {csv_path.name}: {e}")

    return tables

In [ ]:
tabular_dir = data_folder + "/tabular"
tables = read_tabular_folder(tabular_dir)

weather_zones = tables["hex05_WeatherZones"]
daily_weather = tables["hex05_DailyWeather"]
ignition_distribution = tables["hex05_IgnitionDistribution"]

for name, df in tables.items():
    print("\n" + "=" * 80)
    print(name)
    print(df.head())

In [ ]:
daily_weather = tables["hex05_DailyWeather"]
print(daily_weather.head())

In [ ]:
# Output: Burn Probability
bp_raster = data_folder + "/results/burnP3Plus_OutputBurnProbability/burnProbability-sn320.tif"
plot_raster_with_masks(raster_path=bp_raster, actual_mask_path=actual_path, buffer_mask_path=buffer_path)

bp_raster = data_folder + "/results/burnP3Plus_OutputBurnProbability/burnProbability-sn319.tif"
plot_raster_with_masks(raster_path=bp_raster, actual_mask_path=actual_path, buffer_mask_path=buffer_path)

bp_raster = data_folder + "/results/burnP3Plus_OutputBurnProbability/burnProbability-sn2.tif"
plot_raster_with_masks(raster_path=bp_raster, actual_mask_path=actual_path, buffer_mask_path=buffer_path)

In [ ]:
# Output: Fire Intensity
bp_raster = data_folder + "/results/burnP3Plus_OutputFireIntensitySummaryMap/fbpSummary-FireIntensity-Median.tif"
plot_raster_with_masks(raster_path=bp_raster, actual_mask_path=actual_path, buffer_mask_path=buffer_path)

bp_raster = data_folder + "/results/burnP3Plus_OutputFireIntensitySummaryMap/fbpSummary-FireIntensity-Average.tif"
plot_raster_with_masks(raster_path=bp_raster, actual_mask_path=actual_path, buffer_mask_path=buffer_path)

In [ ]:
# Output: Rate of Spread
bp_raster = data_folder + "/results/burnP3Plus_OutputRateOfSpreadSummaryMap/fbpSummary-RateOfSpread-Median.tif"
plot_raster_with_masks(raster_path=bp_raster, actual_mask_path=actual_path, buffer_mask_path=buffer_path)

bp_raster = data_folder + "/results/burnP3Plus_OutputRateOfSpreadSummaryMap/fbpSummary-RateOfSpread-Average.tif"
plot_raster_with_masks(raster_path=bp_raster, actual_mask_path=actual_path, buffer_mask_path=buffer_path)